# A frame, from design to site: cut list, weight, erection sequence

The stud wall was flat, and a topology rule was enough. This is a small
shed in 3D: sills, posts, plates, tie beams, rafters, a ridge, braces. Three
things get harder, and the model graph is what answers each of them:

- several beams meet **in one point**, and a rule made for pairs starts to
  give uneven results
- the workshop wants a **cut list** with as few different parts as possible
- the site wants to know **what goes up first**

> **Needs the timber extra**: `uv run --extra timber jupyter lab`. Units
> are millimetres. Do `08_timber_stud_wall` first.

In [ ]:
# Setup: run this cell first. It finds the shared drawing helper (viz.py),
# whether this notebook sits in Lecture/notebooks/ or is your copy in MyWork/.
import sys
import pathlib

HERE = pathlib.Path.cwd()
NB = next(p / "Lecture" / "notebooks" for p in [HERE, *HERE.parents]
          if (p / "Lecture" / "notebooks" / "viz.py").exists())
sys.path.insert(0, str(NB))
from viz import chart, draw_graph, draw_mesh, draw_plan, draw_points, ramp, row, view3d

from collections import Counter, deque

from compas.colors import Color
from compas.geometry import Line
from compas.tolerance import Tolerance
from compas_timber.connections import JointTopology, LButtJoint, TButtJoint
from compas_timber.elements import Beam
from compas_timber.model import TimberModel

## Step 1 — the shed is a function
`bays` frames along x, a pitched roof across y. The name of every line
starts with its role, which is all the classification this model needs.

In [ ]:
def shed(bays=2, bay=2400, depth=3600, height=2400, rise=1200, knee=800):
    """{name: Line} for a gabled shed; the role is the part of the name before '_'."""
    length = bays * bay
    lines = {}
    for side, y in (("s", 0), ("n", depth)):
        lines[f"sill_{side}"] = Line([0, y, 0], [length, y, 0])
        lines[f"plate_{side}"] = Line([0, y, height], [length, y, height])
        for i in range(bays + 1):
            lines[f"post_{side}{i}"] = Line([i * bay, y, 0], [i * bay, y, height])
        lines[f"brace_{side}w"] = Line([0, y, height - knee], [knee, y, height])
        lines[f"brace_{side}e"] = Line([length, y, height - knee], [length - knee, y, height])
    for i in range(bays + 1):
        x = i * bay
        lines[f"tie_{i}"] = Line([x, 0, height], [x, depth, height])
        lines[f"rafter_s{i}"] = Line([x, 0, height], [x, depth / 2, height + rise])
        lines[f"rafter_n{i}"] = Line([x, depth, height], [x, depth / 2, height + rise])
    lines["ridge"] = Line([0, depth / 2, height + rise], [length, depth / 2, height + rise])
    return lines


def role(name):
    return name.split("_")[0]


lines = shed()
print(len(lines), "members:", dict(Counter(role(name) for name in lines)))
row(draw_points([], "side: x across, z up", size=(480, 330),
                lines=[((l.start.x, l.start.z), (l.end.x, l.end.z)) for l in lines.values()],
                bounds=[(0, 0), (4800, 3600)]),
    draw_points([], "gable: y across, z up", size=(400, 330),
                lines=[((l.start.y, l.start.z), (l.end.y, l.end.z)) for l in lines.values()],
                bounds=[(0, 0), (3600, 3600)]))

## Step 2 — a section per role
Square posts and plates, deep and narrow ties and rafters, light braces.
The section is a property of the **role**, so it is one dictionary.

In [ ]:
SECTION = {"sill": (100, 100), "post": (100, 100), "plate": (100, 100),
           "tie": (60, 160), "rafter": (60, 160), "ridge": (60, 160), "brace": (60, 100)}


def frame_model(lines):
    model = TimberModel(tolerance=Tolerance(unit="MM", absolute=1e-3))
    for name, line in lines.items():
        beam = Beam.from_centerline(line, *SECTION[role(name)])
        beam.name = name
        beam.attributes["category"] = role(name)
        model.add_element(beam)
    model.connect_adjacent_beams()
    return model


model = frame_model(lines)
palette = dict(zip(SECTION, ["#868e96", "#1c7ed6", "#868e96", "#2f9e44", "#e8590c", "#7048e8", "#f59f00"]))
view3d(*[(beam.blank, {"color": Color.from_hex(palette[beam.attributes["category"]])}) for beam in model.beams])

## Step 3 — crowded nodes
How do the members meet? And **how many meet in the same place?** Group
the candidates by their location: a point shared by 6 candidates is a point
where 4 beams meet (4 beams make 6 pairs).

In [ ]:
NAMES = {JointTopology.TOPO_L: "L", JointTopology.TOPO_T: "T", JointTopology.TOPO_X: "X"}
candidates = list(model.joint_candidates)
print(len(candidates), "candidates:", dict(Counter(NAMES[c.topology] for c in candidates)))

at_point = Counter(tuple(round(c) for c in candidate.location) for candidate in candidates)
print("candidates per meeting point:", dict(sorted(Counter(at_point.values()).items())))
busiest = max(at_point, key=at_point.get)
print(f"at {busiest}:", sorted({e.name for c in candidates for e in c.elements
                                 if tuple(round(v) for v in c.location) == busiest}))

Six eaves corners, each with a post, a plate, a tie and a rafter arriving at
one point. On a drawing that is one node. For compas_timber it is six
separate pairs, each of which will get its own joint.

## Step 4 — the rule from the stud wall, and what it does here
L becomes a butt joint, T becomes a butt joint (main beam checked, as
before). For an L joint the wall never cared *which* beam stops and which
runs past. Take the pair in the order the model hands it over.

In [ ]:
def ends_at(a, b, tol=1.0):
    return any(b.centerline.closest_point(end).distance_to_point(end) < tol + max(b.width, b.height) / 2
               for end in (a.centerline.start, a.centerline.end))


def join(model, corner_order):
    """Butt joints everywhere. corner_order(a, b) -> (the beam that stops, the beam that runs past)."""
    for candidate in list(model.joint_candidates):
        a, b = candidate.elements
        if candidate.topology == JointTopology.TOPO_T:
            main, cross = (a, b) if ends_at(a, b) else (b, a)
            TButtJoint.create(model, main, cross)
        elif candidate.topology == JointTopology.TOPO_L:
            LButtJoint.create(model, *corner_order(a, b))
    return model.process_joinery()


def cut_list(model):
    """{(role, section, blank length in mm): how many}"""
    return Counter((beam.attributes["category"], f"{beam.width}x{beam.height}", round(beam.blank_length))
                   for beam in model.beams)


def print_cut_list(model):
    for (name, section, length), count in sorted(cut_list(model).items()):
        print(f"   {count} x {name:7s} {section:8s} {length:5d} mm")


errors = join(model, corner_order=lambda a, b: (a, b))
print(len(errors), "joinery errors;", len(cut_list(model)), "different parts:")
print_cut_list(model)

No errors, and a poor result. Six posts that are identical on the drawing
come out in **several different lengths**, and so do the ties and the
rafters. Nothing is wrong with any single joint. At a crowded node each pair
was decided on its own, in whatever order the pairs happened to come, and
every decision lengthened or shortened somebody.

Restart the kernel and run it again: the list can come out differently,
because the order of the pairs is not fixed. **A result that depends on the
order of a loop is the clearest sign that a rule is missing.**

## Step 5 — a better rule: decide who runs through
A carpenter does not decide a corner pair by pair. There is an order of
precedence: the sill runs through, the plate runs through, the post stops
under the plate, the rafter stops against whatever it lands on. Write that
order down, and let the **lower-ranked beam stop**.

In [ ]:
RANK = ["sill", "plate", "ridge", "post", "tie", "rafter", "brace"]        # first = runs through


def by_rank(a, b):
    stops, runs = sorted((a, b), key=lambda beam: RANK.index(beam.attributes["category"]), reverse=True)
    return stops, runs


ranked = frame_model(lines)
errors = join(ranked, corner_order=by_rank)
print(len(errors), "joinery errors;", len(cut_list(ranked)), "different parts:")
print_cut_list(ranked)

Same centrelines, same joints, same number of cuts. About half as many
different parts, all six posts are one line of the cutting list, and the
answer is the same on every run. **The design knowledge was one list of
seven words.**

Read the lengths with care: they come from butt joints stacked on one
point, which no carpenter would build. A real eaves node is offset and
detailed (the plate sits on the post, the tie on the plate, the rafter is
notched over both), and that detailing needs Rhino. What carries over is
the method: make the rule explicit, then count what it produces.

## Step 6 — volume, weight, and the heaviest piece
The blank of each beam is a box, so its volume is three numbers. At about
450 kg/m³ for spruce, the cut list becomes a weight list: what the truck
carries, and what two people can lift.

In [ ]:
DENSITY = 450                                             # kg per m3

mass = {beam.name: beam.blank_length * beam.width * beam.height / 1e9 * DENSITY for beam in ranked.beams}
by_role = Counter()
for beam in ranked.beams:
    by_role[beam.attributes["category"]] += mass[beam.name]
for name, kg in by_role.most_common():
    print(f"{name:7s} {kg:6.1f} kg")
print(f"total   {sum(mass.values()):6.1f} kg in {sum(mass.values()) / DENSITY:.2f} m3 of timber")
heaviest = max(mass, key=mass.get)
print(f"heaviest piece: {heaviest}, {mass[heaviest]:.1f} kg")

## Step 7 — what goes up first?
The model graph again. Two questions:

1. **Is it one structure?** One connected component, or did a member end up
   joined to nothing?
2. **In what order can it be erected?** The sills lie on the foundation. A
   beam can go up once something it joins is standing. That is
   breadth-first search from all the sills at once: the same search as the
   escape distances in `02_floor_plan`, with sills for exits.

In [ ]:
graph = ranked.graph
name_of = {beam.graphnode: beam.name for beam in ranked.beams}
print("connected components:", len(graph.connected_nodes()))

stage = {n: 0 for n in graph.nodes() if name_of[n].startswith("sill")}
queue = deque(stage)
while queue:
    n = queue.popleft()
    for other in graph.neighbors(n):
        if other not in stage:
            stage[other] = stage[n] + 1
            queue.append(other)

last = max(stage.values())
for s in range(last + 1):
    members = sorted(name_of[n] for n in stage if stage[n] == s)
    kilos = sum(mass[name] for name in members)
    print(f"stage {s}: {len(members):2d} members, {kilos:5.1f} kg   {Counter(role(name) for name in members).most_common()}")

In [ ]:
stage_of = {name_of[n]: s for n, s in stage.items()}
view3d(*[(beam.blank, {"color": Color.from_hex(ramp(stage_of[beam.name] / last))}) for beam in ranked.beams])

Blue goes up first, orange last. Treat it as a **first draft** of a
sequence: the search knows what touches what, and nothing about gravity or
temporary bracing. It puts the rafters in the same stage as the plates,
because a rafter touches a post. A site manager would hold them back until
the ties are in. Compare with the brick wall (`05_brick_wall_assembly`), where *rests on*
was a directed edge and the sequence was strict.

## Step 8 — the fabrication file, and the design as a parameter

In [ ]:
import xml.etree.ElementTree as ET

from compas_timber.fabrication import BTLxWriter


def machining(model):
    counts = Counter()
    for element in ET.fromstring(BTLxWriter().model_to_xml(model)).iter():
        if element.tag.endswith("Processings"):
            counts.update(child.tag.split("}")[-1] for child in element)
    return counts


print("BTLx processings:", dict(machining(ranked)))

print(f"\n{'bays':>4s} {'members':>8s} {'joints':>7s} {'parts':>6s} {'cuts':>5s} {'timber':>8s} {'weight':>8s}")
for bays in (1, 2, 3, 4):
    variant = frame_model(shed(bays=bays))
    join(variant, corner_order=by_rank)
    beams = list(variant.beams)
    volume = sum(beam.blank_length * beam.width * beam.height for beam in beams) / 1e9
    print(f"{bays:4d} {len(beams):8d} {len(list(variant.joints)):7d} {len(cut_list(variant)):6d} "
          f"{sum(machining(variant).values()):5d} {volume:6.2f}m3 {volume * DENSITY:6.0f}kg")

Members, joints, cuts and weight all grow with the number of bays. The
number of **different** parts does not: a longer shed is more of the same
pieces. That is what an explicit rule buys.

## Your turn
1. Put `"post"` before `"plate"` in `RANK` (posts run through, plates stop
   between them). What happens to the cut list? Which version would you
   rather build?
2. The rafters still come in two lengths. Find the joint responsible, and
   try `LMiterJoint` for rafter–rafter pairs only.
3. Give the erection search a rule of its own: a rafter may only go up
   after its tie. Which data structure from `05_brick_wall_assembly` do you need?
4. Remove the four braces and rerun Step 7. Does the graph notice? Should
   it? What kind of analysis would?